In [54]:
import os 
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

In [55]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

In [56]:
from langchain_core.messages import HumanMessage

In [57]:
llm_model = ChatGroq(
    model= "llama-3.3-70b-versatile",
    temperature= 0.5,
    max_retries=2,
)

In [58]:
#Prompt Template 
prompt_1 = PromptTemplate(
    input_variables = ['restaurant_name'],
    template = "I want to open a restaurant for {restaurant_name} food. Suggest a fancy name for this, only one name, no explination."
)
prompt_1.format(restaurant_name="Indian")

'I want to open a restaurant for Indian food. Suggest a fancy name for this, only one name, no explination.'

In [59]:
llm_name_chain = prompt_1 | llm_model | StrOutputParser()
llm_name_chain.invoke({"restaurant_name" : "arabic"})

'Sahara Nights'

In [60]:
#Prompt Template 2 

prompt_2 = PromptTemplate(
    input_variables= ['menu'],
    template= """Suggest menu items for {menu} 12. Return ONLY a comma-separated list, no explanations, no translations."""
)

prompt_2.format(menu="arabic")

'Suggest menu items for arabic 12. Return ONLY a comma-separated list, no explanations, no translations.'

In [61]:
llm_menu_chain = prompt_2 | llm_model | StrOutputParser()
llm_menu_chain.invoke({"menu" : "arabic"})

'Shawarma, Falafel, Kebab, Machboos, Gormeh, Fattoush, Tabouleh, Kibbeh, Baba Ghanoush, Hummus, Muhalabia, Umm Ali'

In [62]:
llm_model = llm_model

prompt_1 = PromptTemplate(
    input_variables = ['restaurant_name'],
    template = "I want to open a restaurant for {restaurant_name} food. Suggest a fancy name for this, only one name, no explination."
)
llm_name_chain = prompt_1 | llm_model | StrOutputParser()

prompt_2 = PromptTemplate(
    input_variables= ['menu'],
    template= """Suggest menu items for {menu} 12. Return ONLY a comma-separated list, no explanations, no translations."""
)
llm_menu_chain = prompt_2 | llm_model | StrOutputParser()


final_chain = llm_name_chain | llm_menu_chain | StrOutputParser()
final_chain.invoke({"restaurant_name" : "arabic"})

'Machboos, Gormeh Sabzi, Shawarma, Falafel, Kebabs, Fattoush, Hummus, Tabouleh, Mixed Grill, Luqaimat, Umm Ali, Kunafeh'

In [65]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

llm_name_chain = prompt_1 | llm_model | StrOutputParser()

llm_menu_chain = (RunnableLambda(lambda resname: {"menu" : resname})) | prompt_2 | llm_model | StrOutputParser()

final_chain = llm_name_chain | llm_menu_chain | StrOutputParser()
final_chain.invoke({"restaurant_name" : "arabic"})



"Shawarma, Falafel, Hummus, Tabouleh, Fattoush, Kebabs, Gyro, Baba Ghanoush, Dolmas, Ma'amoul, Baklava, Umm Ali"

In [70]:

# Final Chain
final_chain = (llm_name_chain | RunnableParallel(restaurant_name=RunnableLambda(lambda x: x), menu=RunnableLambda(lambda x: {"menu": x}) | llm_menu_chain,))
result = final_chain.invoke({"restaurant_name": "arabic"})

print(result)

{'restaurant_name': 'Sahara Nights', 'menu': "Shawarma, Falafel, Hummus, Tabouleh, Grilled Halloumi, Chicken Kebabs, Lamb Tagine, Couscous, Baklava, Ma'amoul, Ghoriba, Umm Ali"}


invoke({"restaurant_name":"arabic"})
                │
                ▼
        llm_name_chain
                │
                ▼
        "Desert Oasis"
                │
                ▼
         RunnableParallel
        ┌─────────┴─────────┐
        │                   │
        ▼                   ▼
 lambda x:x        lambda x:{"menu":x}
        │                   │
        ▼                   ▼
"Desert Oasis"      {"menu":"Desert Oasis"}
                            │
                            ▼
                     llm_menu_chain
                            │
                            ▼
        "Hummus,Falafel,Shawarma..."
        └─────────┬─────────┘
                  ▼
{
    "restaurant_name":"Desert Oasis",
    "menu":"Hummus,Falafel,Shawarma..."
}


RunnableParallel does not send the output of one branch into the other. Instead, it sends the same input to both branches at the same time, 
waits for both to finish, and then gathers their outputs into a dictionary using the names you provided (restaurant_name and menu). 
This is why you get both the generated restaurant name and the generated menu back from a single invoke() call.